# Automatización de Notificaciones de Traslados BDUA

## Descripción General
Este notebook automatiza el proceso completo de **notificación masiva** relacionado con traslados de salida (aprobados y negados) en procesos BDUA. Abarca desde la generación de documentos hasta el **envío automatizado de correos electrónicos masivos**, gestionando comunicaciones multicanal desde la EPS de origen hacia los usuarios finales.

## Objetivos
- **Enviar correos electrónicos masivos** de forma automatizada con notificaciones personalizadas por usuario
- Extraer y gestionar **números de teléfono** para notificaciones vía SMS
- Crear **reportes en PDF** individualizados adjuntos a cada notificación
- Consolidar información en **archivo Excel** para seguimiento y trazabilidad

## Insumos de Entrada
| Insumo | Descripción |
|--------|-------------|
| Base de traslados BDUA | Registros de traslados aprobados y negados |
| Información de usuarios | Correos electrónicos, números de teléfono |
| Datos EPS de origen | Información complementaria de las EPS |

## Productos de Salida
1. **Correos masivos enviados**: Notificaciones automáticas con PDF adjunto por cada usuario
2. **PDF por usuario**: Documentos de notificación personalizados con detalle del traslado
3. **Excel consolidado**: Tabla de seguimiento con:
   - Información del usuario
   - Números de teléfono y correos electrónicos
   - Estado del traslado
   - Registro de notificaciones enviadas

## Estructura del Notebook
1. Carga y validación de datos fuente
2. Procesamiento de información de usuarios
3. Generación de PDF de notificaciones
4. Extracción de contactos (email y teléfono)
5. **Envío automatizado de correos masivos** con PDF adjunto
6. Consolidación de resultados en archivo Excel
7. Reportes de ejecución y trazabilidad

# Mudolos

In [ ]:
# Módulos necesarios
import os  # Trabajar con rutas del sistema
import pandas as pd  # Trabajar con DataFrames
import numpy as np # Operaciones numéricas y manejo de datos faltantes
import datetime  # Manejo de fechas
from datetime import datetime  # Para convertir cadenas a objetos de fecha
from pathlib import Path  # Manejo de rutas
import smtplib  # Envío de correos
from email.mime.text import MIMEText  # Crear correos con texto
from email.mime.multipart import MIMEMultipart  # Correos con múltiples partes
from PyPDF2 import PdfReader  # Leer archivos PDF
import re  # Para validación de correos electrónicos
import gc # Recolección de basura para liberar memoria
from typing import List # Para anotaciones de tipos en funciones

# Rutas y Variables

In [ ]:
# --- Lógica de Rutas Dinámicas ---
# Obtenemos la ruta base del perfil de usuario (ej: C:\Users\NombreUsuario)
user_profile = os.environ['USERPROFILE']

# Definimos la parte común de la ruta de OneDrive para evitar repeticiones (Principio DRY)
# Nota: Si el nombre de la carpeta de OneDrive varía entre PCs, podemos mejorar esto.
base_onedrive = os.path.join(user_profile, "OneDrive - 891856000_CAPRESOCA E P S", "Capresoca", "AlmostClear")

# --- Variables de Control ---
Periodo_notificación = "01/04/2026"
Fecha_Correo = "01/05/2026"
Informe = "Informe  #05"

# --- Definición de Rutas de Entrada ---
# Usamos os.path.join para que sea compatible con cualquier sistema operativo y maneje bien las barras inclinadas
R_s1_automatico = os.path.join(base_onedrive, "Procesos BDUA", "Subsidiados", "Procesos BDUA EPS", "Automatico-S1", "All-AUTO-S1.txt")
R_s1_val = os.path.join(base_onedrive, "Procesos BDUA", "Subsidiados", "Procesos BDUA EPS", "S1.val", "All-S1-VAL.txt")
R_s5 = os.path.join(base_onedrive, "Procesos BDUA", "Subsidiados", "Procesos BDUA EPS", "S5", "S5 TXT", "S5_consolidado.txt")

# Maestro SIE
R_maestro_sie = os.path.join(base_onedrive, "SIE", "Aseguramiento", "ms_sie", "Reporte_Validación Archivos Maestro_2026_04_28.csv")

# Codigo Dane
R_codigo_dane = os.path.join(base_onedrive, "Constantes", "Departamentos.txt")

# Correos EPS
R_correos_eps = os.path.join(base_onedrive, "Constantes", "CORREO DE OTRAS EPS.xlsx")
Hoja_correos_eps = "correos eps"

# --- Ruta de Salida ---
# Construcción dinámica para el escritorio y la estructura de carpetas del contrato
R_salida_base = os.path.join(user_profile, "OneDrive - 891856000_CAPRESOCA E P S", "Escritorio", "Yesid Rincón Z", "informes", "2026", "CTO 102.2026", f"CTO102.2026 {Informe}", "12 Actividad", "Bases de datos notificaciones telefonicas", "TRASLADOS APROBADOS Y NEGADOS BDUA")

# Verificación de existencia (Opcional pero recomendado para evitar errores de ejecución)
if not os.path.exists(base_onedrive):
    print(f"⚠️ Alerta: No se encontró la ruta base de OneDrive en: {base_onedrive}")

# Cargue de dataframes

In [ ]:
df_s1_automatico = pd.read_csv(R_s1_automatico, sep=",", encoding="latin-1", dtype=str)
df_s1_val = pd.read_csv(R_s1_val, sep=",", encoding="latin-1", dtype=str)
df_s5 = pd.read_csv(R_s5, sep=",", encoding="latin-1", dtype=str)

df_ms_sie = pd.read_csv(R_maestro_sie, sep=';', encoding='ANSI', header=0, dtype=str)
df_codigo_dane = pd.read_csv(R_codigo_dane, sep=';', encoding='utf-8', header=0, dtype=str)

df_eps = pd.read_excel(
    R_correos_eps,                                  # La ruta de tu archivo
    sheet_name=Hoja_correos_eps,                  # El nombre exacto de la hoja
    usecols=['CORREO', 'ENTIDAD', 'COD ENTIDAD'],   # Solo cargamos estas 3 columnas
    dtype=str                                       # Forzamos TODO a formato texto
)


# Limpieza de datos

## Procesos BDUA

### Fecha a reportar

In [ ]:
# Extraer mes y año del periodo de notificación
mes_filtro = int(Periodo_notificación.split("/")[1])  # Mes
anio_filtro = int(Periodo_notificación.split("/")[2])  # Año

print(f"{'='*60}")
print(f"FILTRO APLICADO: Mes={mes_filtro:02d} | Año={anio_filtro}")
print(f"{'='*60}\n")

# Función para filtrar por mes y año y mostrar resumen
def filtrar_por_periodo(df, nombre_df, col_fecha="FECHA_PROCESO"):
    """Filtra el DataFrame por mes/año y muestra resumen en consola."""
    registros_antes = len(df)
    
    # Convertir la columna de fecha a datetime para extraer mes y año
    fecha_parsed = pd.to_datetime(df[col_fecha], format="%d/%m/%Y", dayfirst=True)
    
    # Filtrar por mes y año
    mask = (fecha_parsed.dt.month == mes_filtro) & (fecha_parsed.dt.year == anio_filtro)
    df_filtrado = df[mask].copy()
    
    registros_despues = len(df_filtrado)
    
    # Reporte en consola
    print(f"📋 {nombre_df}")
    print(f"   Registros antes: {registros_antes:,} → Después: {registros_despues:,}")
    print(f"   Distribución por FECHA_PROCESO:")
    
    conteo_fechas = df_filtrado[col_fecha].value_counts().sort_index()
    for fecha, cantidad in conteo_fechas.items():
        print(f"     • {fecha}: {cantidad:,} registros")
    
    print(f"   Total fechas únicas: {len(conteo_fechas)}")
    print(f"{'-'*60}")
    
    return df_filtrado

# Aplicar filtro a los 3 DataFrames
df_s1_automatico = filtrar_por_periodo(df_s1_automatico, "df_s1_automatico")
df_s1_val = filtrar_por_periodo(df_s1_val, "df_s1_val")
df_s5 = filtrar_por_periodo(df_s5, "df_s5")

# Resumen final
print(f"\n{'='*60}")
print(f"RESUMEN FINAL DEL FILTRO")
print(f"{'='*60}")
print(f"  df_s1_automatico : {len(df_s1_automatico):>8,} registros")
print(f"  df_s1_val        : {len(df_s1_val):>8,} registros")
print(f"  df_s5            : {len(df_s5):>8,} registros")
print(f"  {'─'*40}")
print(f"  TOTAL            : {len(df_s1_automatico) + len(df_s1_val) + len(df_s5):>8,} registros")

### Filtrar Traslados

In [ ]:
# Filtrar traslados de EPS (eliminar movilidades)
print(f"{'='*60}")
print(f"FILTRO: Traslados de EPS (eliminar movilidades)")
print(f"  Excluir TIPO_TRASLADO: 3, 4, 5")
print(f"  Excluir ENT_ID_ORIGEN: EPS025, EPSC25")
print(f"{'='*60}\n")

tipos_excluir = ["3", "4", "5"]
entidades_excluir = ["EPS025", "EPSC25"]

def filtrar_traslados_eps(df, nombre_df):
    """Filtra movilidades y entidades propias, muestra resumen."""
    registros_antes = len(df)
    
    # Aplicar filtros: excluir tipos de traslado y entidades
    mask = (
        ~df["TIPO_TRASLADO"].isin(tipos_excluir) &
        ~df["ENT_ID_ORIGEN"].isin(entidades_excluir)
    )
    df_filtrado = df[mask].copy()
    
    registros_despues = len(df_filtrado)
    eliminados = registros_antes - registros_despues
    
    # Reporte en consola
    print(f"📋 {nombre_df}")
    print(f"   Registros antes: {registros_antes:,} → Después: {registros_despues:,} (eliminados: {eliminados:,})")
    
    print(f"\n   Distribución por TIPO_TRASLADO:")
    conteo_tipo = df_filtrado["TIPO_TRASLADO"].value_counts().sort_index()
    for tipo, cantidad in conteo_tipo.items():
        print(f"     • Tipo {tipo}: {cantidad:,} registros")
    
    print(f"\n   Distribución por ENT_ID_ORIGEN:")
    conteo_entidad = df_filtrado["ENT_ID_ORIGEN"].value_counts().sort_index()
    for entidad, cantidad in conteo_entidad.items():
        print(f"     • {entidad}: {cantidad:,} registros")
    
    print(f"{'-'*60}")
    
    return df_filtrado

# Aplicar filtro a los 2 DataFrames
df_s1_automatico = filtrar_traslados_eps(df_s1_automatico, "df_s1_automatico")
df_s1_val = filtrar_traslados_eps(df_s1_val, "df_s1_val")

# Resumen final
print(f"\n{'='*60}")
print(f"RESUMEN DESPUÉS DE FILTRAR TRASLADOS EPS")
print(f"{'='*60}")
print(f"  df_s1_automatico : {len(df_s1_automatico):>8,} registros")
print(f"  df_s1_val        : {len(df_s1_val):>8,} registros")
print(f"  df_s5            : {len(df_s5):>8,} registros (sin cambios)")
print(f"  {'─'*40}")
print(f"  TOTAL            : {len(df_s1_automatico) + len(df_s1_val) + len(df_s5):>8,} registros")

### Unificar información en un solo datafarme S1.val

In [ ]:
# ============================================================
# CONSOLIDACIÓN DE TRASLADOS DE ENTRADA (Aprobados y Negados)
# ============================================================

print(f"{'='*70}")
print(f"CONSOLIDACIÓN DE TRASLADOS DE ENTRADA")
print(f"  S1-Automático: Aprobados automáticos por ADRES")
print(f"  S5: Respuesta EPS origen (0=Negado, 1=Aprobado)")
print(f"  S1-Val: Base maestra con información completa")
print(f"{'='*70}\n")

# Columnas clave para el cruce
cols_clave = ["TPS_IDN_ID", "HST_IDN_NUMERO_IDENTIFICACION", "FECHA_PROCESO"]

# ── PASO 1: Marcar aprobados automáticos desde S1-Automático ──
print(f"{'─'*70}")
print(f"PASO 1: Identificar aprobados automáticos (S1-Automático)")
print(f"{'─'*70}")

# Crear set de claves del S1-Automático para búsqueda eficiente
claves_s1_auto = set(
    df_s1_automatico[cols_clave].apply(lambda x: tuple(x), axis=1)
)
print(f"  Registros únicos en S1-Automático: {len(claves_s1_auto):,}")

# ── PASO 2: Preparar S5 (respuesta EPS origen) ──
print(f"\n{'─'*70}")
print(f"PASO 2: Preparar respuestas del S5")
print(f"{'─'*70}")

# Crear DataFrame de respuestas S5 con las columnas clave + RESPUESTA
df_s5_respuesta = df_s5[cols_clave + ["RESPUESTA"]].copy()
df_s5_respuesta["RESPUESTA"] = df_s5_respuesta["RESPUESTA"].str.strip()

conteo_s5 = df_s5_respuesta["RESPUESTA"].value_counts()
print(f"  Total registros S5: {len(df_s5_respuesta):,}")
for resp, cant in conteo_s5.items():
    etiqueta = "Aprobado" if resp == "1" else "Negado" if resp == "0" else f"Desconocido ({resp})"
    print(f"    • {etiqueta} (RESPUESTA={resp}): {cant:,}")

# Crear set de claves S5 para búsqueda
claves_s5 = set(
    df_s5_respuesta[cols_clave].apply(lambda x: tuple(x), axis=1)
)
print(f"  Registros únicos S5 por clave: {len(claves_s5):,}")

# ── PASO 3: Cruzar S1-Val con S1-Automático y S5 ──
print(f"\n{'─'*70}")
print(f"PASO 3: Cruzar S1-Val con S1-Automático y S5")
print(f"{'─'*70}")

df_consolidado = df_s1_val.copy()
registros_s1_val = len(df_consolidado)
print(f"  Registros en S1-Val: {registros_s1_val:,}")

# Crear tupla de claves en S1-Val
df_consolidado["_clave"] = list(
    df_consolidado[cols_clave].apply(lambda x: tuple(x), axis=1)
)

# Marcar origen: S1-Automático o S5
df_consolidado["ORIGEN_RESPUESTA"] = df_consolidado["_clave"].apply(
    lambda x: "S1-AUTOMATICO" if x in claves_s1_auto 
              else ("S5" if x in claves_s5 else "SIN_RESPUESTA")
)

# Cruzar con S5 para obtener la respuesta (merge por claves)
df_consolidado = df_consolidado.merge(
    df_s5_respuesta[cols_clave + ["RESPUESTA"]],
    on=cols_clave,
    how="left"
)

# Asignar estado del traslado
# S1-Automático → siempre Aprobado
# S5 → según columna RESPUESTA (1=Aprobado, 0=Negado)
# Sin respuesta → marcar como pendiente
df_consolidado["ESTADO_TRASLADO"] = df_consolidado.apply(
    lambda row: "APROBADO" if row["ORIGEN_RESPUESTA"] == "S1-AUTOMATICO"
                else ("APROBADO" if row["RESPUESTA"] == "1" 
                      else ("NEGADO" if row["RESPUESTA"] == "0" 
                            else "SIN_RESPUESTA")),
    axis=1
)

# Eliminar columna auxiliar
df_consolidado.drop(columns=["_clave"], inplace=True)

# Reporte de clasificación
print(f"\n  Clasificación por ORIGEN_RESPUESTA:")
conteo_origen = df_consolidado["ORIGEN_RESPUESTA"].value_counts()
for origen, cant in conteo_origen.items():
    print(f"    • {origen}: {cant:,}")

print(f"\n  Clasificación por ESTADO_TRASLADO:")
conteo_estado = df_consolidado["ESTADO_TRASLADO"].value_counts()
for estado, cant in conteo_estado.items():
    print(f"    • {estado}: {cant:,}")

# ── PASO 4: Deduplicación priorizando aprobados ──
print(f"\n{'─'*70}")
print(f"PASO 4: Deduplicación (priorizar aprobados sobre negados)")
print(f"{'─'*70}")

registros_antes_dedup = len(df_consolidado)

# Columnas de identificación del usuario (sin FECHA_PROCESO)
cols_usuario = ["TPS_IDN_ID", "HST_IDN_NUMERO_IDENTIFICACION"]

# Crear prioridad: APROBADO > NEGADO > SIN_RESPUESTA
prioridad_estado = {"APROBADO": 0, "NEGADO": 1, "SIN_RESPUESTA": 2}
df_consolidado["_prioridad"] = df_consolidado["ESTADO_TRASLADO"].map(prioridad_estado)

# Ordenar: por usuario, prioridad (aprobado primero), fecha proceso (más reciente primero)
df_consolidado["_fecha_orden"] = pd.to_datetime(
    df_consolidado["FECHA_PROCESO"], format="%d/%m/%Y", dayfirst=True
)
df_consolidado.sort_values(
    by=cols_usuario + ["_prioridad", "_fecha_orden"],
    ascending=[True, True, True, False],  # Prioridad ascendente (0=aprobado primero), fecha descendente
    inplace=True
)

# Mantener el primer registro por usuario (el de mayor prioridad y fecha más reciente)
df_consolidado.drop_duplicates(subset=cols_usuario, keep="first", inplace=True)

# Eliminar columnas auxiliares
df_consolidado.drop(columns=["_prioridad", "_fecha_orden"], inplace=True)

registros_despues_dedup = len(df_consolidado)
duplicados_eliminados = registros_antes_dedup - registros_despues_dedup

print(f"  Registros antes: {registros_antes_dedup:,}")
print(f"  Registros después: {registros_despues_dedup:,}")
print(f"  Duplicados eliminados: {duplicados_eliminados:,}")

# ── PASO 5: Validación de coherencia temporal ──
print(f"\n{'─'*70}")
print(f"PASO 5: Validación de coherencia temporal")
print(f"{'─'*70}")

print(f"\n  Distribución por FECHA_PROCESO:")
conteo_fechas = df_consolidado["FECHA_PROCESO"].value_counts().sort_index()
for fecha, cant in conteo_fechas.items():
    print(f"    • {fecha}: {cant:,} registros")

print(f"\n  Cruce ESTADO_TRASLADO × ORIGEN_RESPUESTA:")
tabla_cruzada = pd.crosstab(
    df_consolidado["ESTADO_TRASLADO"], 
    df_consolidado["ORIGEN_RESPUESTA"],
    margins=True,
    margins_name="TOTAL"
)
print(tabla_cruzada.to_string(col_space=15))

print(f"\n  Distribución ESTADO × FECHA_PROCESO:")
tabla_fecha_estado = pd.crosstab(
    df_consolidado["FECHA_PROCESO"], 
    df_consolidado["ESTADO_TRASLADO"],
    margins=True,
    margins_name="TOTAL"
)
print(tabla_fecha_estado.to_string(col_space=12))

# ── RESUMEN FINAL ──
print(f"\n{'='*70}")
print(f"RESUMEN FINAL - CONSOLIDADO DE TRASLADOS DE ENTRADA")
print(f"{'='*70}")
print(f"  Total registros consolidados : {len(df_consolidado):>8,}")
print(f"  ├── APROBADOS                : {len(df_consolidado[df_consolidado['ESTADO_TRASLADO'] == 'APROBADO']):>8,}")
print(f"  │   ├── S1-Automático        : {len(df_consolidado[(df_consolidado['ESTADO_TRASLADO'] == 'APROBADO') & (df_consolidado['ORIGEN_RESPUESTA'] == 'S1-AUTOMATICO')]):>8,}")
print(f"  │   └── S5                   : {len(df_consolidado[(df_consolidado['ESTADO_TRASLADO'] == 'APROBADO') & (df_consolidado['ORIGEN_RESPUESTA'] == 'S5')]):>8,}")
print(f"  ├── NEGADOS                  : {len(df_consolidado[df_consolidado['ESTADO_TRASLADO'] == 'NEGADO']):>8,}")
print(f"  └── SIN RESPUESTA            : {len(df_consolidado[df_consolidado['ESTADO_TRASLADO'] == 'SIN_RESPUESTA']):>8,}")
print(f"  {'─'*50}")
print(f"  Usuarios únicos              : {df_consolidado[cols_usuario].drop_duplicates().shape[0]:>8,}")
print(f"  Fechas de proceso            : {df_consolidado['FECHA_PROCESO'].nunique():>8,}")

### Eliminar dataframes, liberar espacio en RAM

In [ ]:
del df_s1_automatico, df_s1_val, df_s5  # Liberar memoria de DataFrames originales

## Depurar df_consolidado

### Eliminar Columnas

In [ ]:
# Eliminar columnas innecesarias del consolidado
columnas_a_eliminar = [
    "ENT_ID", "TPS_IDN_ID_2", "HST_IDN_NUMERO_IDENTIFICACION_2", "AFL_PRIMER_APELLIDO_2",
    "AFL_SEGUNDO_APELLIDO_2", "AFL_PRIMER_NOMBRE_2", "AFL_SEGUNDO_NOMBRE_2",
    "AFL_FECHA_NACIMIENTO_2", "TPS_GNR_ID_2", "TPS_MDL_SBS_ID"
]

df_consolidado.drop(columns=columnas_a_eliminar, inplace=True, errors="ignore")
print("Columnas innecesarias eliminadas del df_consolidado.")

## Correos y telefono

In [ ]:
import pandas as pd

def cruzar_maestro_sie(df_consolidado, df_ms_sie):
    """
    Realiza el cruce entre el consolidado y el maestro SIE.
    Aplica buenas prácticas de limpieza de llaves y auditoría de datos.
    """
    
    # 1. Estandarización de llaves (Prevención de fallos por espacios o tipos de datos)
    keys_consolidado = ['TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION']
    keys_sie = ['tipo_documento', 'numero_identificacion']
    
    for df, keys in [(df_consolidado, keys_consolidado), (df_ms_sie, keys_sie)]:
        for key in keys:
            df[key] = df[key].astype(str).str.strip()

    # 2. Selección de columnas necesarias del maestro para optimizar memoria
    cols_interes = keys_sie + ['celular', 'telefono_1', 'telefono_2', 'correo_electronico']
    df_ms_sie_subset = df_ms_sie[cols_interes].drop_duplicates(subset=keys_sie)

    # 3. Proceso de Cruce (Left Join)
    # Usamos left join para no perder registros del df_consolidado original
    df_resultado = pd.merge(
        df_consolidado,
        df_ms_sie_subset,
        left_on=keys_consolidado,
        right_on=keys_sie,
        how='left'
    )

    # 4. Auditoría de Calidad
    print("=== REPORTE DE CALIDAD DEL CRUCE ===")
    
    # Cantidad de registros que NO cruzaron (donde las columnas nuevas quedaron NaN)
    # Tomamos 'celular' como referencia, pero lo ideal es validar contra la llave del SIE
    no_cruzaron = df_resultado['tipo_documento'].isna().sum()
    total_registros = len(df_resultado)
    
    print(f"Total registros en consolidado: {total_registros}")
    print(f"Registros que NO se encontraron en SIE: {no_cruzaron} ({(no_cruzaron/total_registros)*100:.2f}%)")
    
    print("\n--- Conteo de datos recuperados (No nulos) ---")
    columnas_nuevas = ['celular', 'telefono_1', 'telefono_2', 'correo_electronico']
    for col in columnas_nuevas:
        conteo = df_resultado[col].notna().sum()
        print(f"Columna '{col}': {conteo} registros con información.")

    # 5. Limpieza post-cruce (opcional: eliminar llaves duplicadas del maestro)
    df_resultado = df_resultado.drop(columns=keys_sie)
    
    return df_resultado, no_cruzaron

# Ejemplo de uso:
df_consolidado, fallos = cruzar_maestro_sie(df_consolidado, df_ms_sie)

### Eliminar df_ms_sie

In [ ]:
del df_ms_sie

## Depurar telefonos

In [ ]:
import pandas as pd
import re

def limpiar_y_validar_telefonos(df, columnas):
    """
    Limpia y valida celulares colombianos en el DataFrame consolidado.
    Distingue entre datos que ya venían vacíos y datos que fueron eliminados por mala calidad.
    """
    reporte = {}

    def es_valido(numero):
        # Si quedó vacío tras la limpieza o era nulo, no es válido
        if not numero or numero == 'nan':
            return False
        # 1. Longitud 10, inicia con 3 y tiene al menos 4 dígitos distintos (Entropía)
        return (len(numero) == 10 and 
                numero.startswith('3') and 
                len(set(numero)) >= 4)

    for col in columnas:
        if col not in df.columns:
            continue
            
        # 1. Contamos cuántos datos REALES (no nulos) hay antes de empezar
        # Esto evita contar los NaN del merge fallido como "datos iniciales"
        datos_reales_iniciales = df[col].dropna().count()
        
        # 2. Limpieza de caracteres
        # Usamos fillna('') antes de convertir a str para evitar el texto "nan"
        df[col] = df[col].fillna('').astype(str).str.replace(r'\D', '', regex=True)
        
        # 3. Aplicamos validación de estructura y calidad
        mask_validos = df[col].apply(es_valido)
        
        # 4. Cálculo de métricas para Lumethik
        validados = mask_validos.sum()
        # Eliminados son los que tenían algo pero no pasaron la regla de calidad
        eliminados = datos_reales_iniciales - validados
        
        # 5. Limpieza final en el DataFrame: lo que no es válido se vuelve NaN real
        df.loc[~mask_validos, col] = None
        
        reporte[col] = {
            'iniciales': datos_reales_iniciales,
            'validados': validados,
            'eliminados': eliminados,
            'vacios_finales': df[col].isna().sum()
        }

    # --- Salida por Consola ---
    print("\n" + "="*80)
    print(f"{'AUDITORÍA DE CALIDAD TELEFÓNICA':^80}")
    print("="*80)
    print(f"{'COLUMNA':<20} | {'EXISTENTES':<12} | {'VALIDADOS':<12} | {'ELIMINADOS':<12} | {'NULOS FINAL'}")
    print("-" * 80)
    
    for col, stats in reporte.items():
        print(f"{col:<20} | {stats['iniciales']:<12} | {stats['validados']:<12} | {stats['eliminados']:<12} | {stats['vacios_finales']}")
    print("="*80 + "\n")
    
    return df

# Aplicación directa sobre tu consolidado
columnas_telefonos = ['celular', 'telefono_1', 'telefono_2']
df_consolidado = limpiar_y_validar_telefonos(df_consolidado, columnas_telefonos)

## Depurar Correos df_consolidado[correo_electronico]

In [ ]:
import pandas as pd
import re

def normalizar_y_validar_correos(df, columna='correo_electronico'):
    """
    Normaliza correos, corrige dominios, elimina correos de relleno 
    (diferenciando coincidencias exactas y parciales) y valida estructura.
    """
    
    correcciones_dominios = {
        r'@gamil\.': '@gmail.',
        r'@gamail\.': '@gmail.',
        r'@gimal\.': '@gmail.',
        r'@gimail\.': '@gmail.',
        r'@hotmial\.': '@hotmail.',
        r'@hotmal\.': '@hotmail.',
        r'@outlok\.': '@outlook.',
        r'@outluk\.': '@outlook.',
        r'@msn\.con$': '@msn.com',
        r'\.con$': '.com',
        r',com$': '.com',
    }

    # 1. Palabras Exactas: Solo se borra si el usuario es EXACTAMENTE esto.
    # Evita borrar "natalia" por culpa de "na".
    palabras_exactas = ['na', 'test', 'prueba', 'correo', 'a', 'x']

    # 2. Palabras Parciales: Se borra si esta palabra aparece EN CUALQUIER LADO.
    palabras_parciales = [
        'actualizar', 'notiene', 'sincorreo', 'noemail', 'noaplica', 'actulizar'
    ]

    def validar_estructura(email):
        if not email or email == 'nan' or email == '':
            return None
            
        parte_usuario = email.split('@')[0] if '@' in email else email
        
        # Validación Exacta
        if parte_usuario in palabras_exactas:
            return None
            
        # Validación Parcial
        if any(palabra in parte_usuario for palabra in palabras_parciales):
            return None

        # Regex estándar para email
        patron = r'^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$'
        if re.match(patron, email):
            return email
            
        return None

    # Proceso de limpieza
    total_inicial = df[columna].dropna().count()
    df[columna] = df[columna].fillna('').astype(str).str.lower().str.strip()
    df[columna] = df[columna].str.replace(',', '.', regex=False)
    
    for error, correccion in correcciones_dominios.items():
        df[columna] = df[columna].str.replace(error, correccion, regex=True)

    mask_validos = df[columna].apply(validar_estructura).notna()
    
    validados = mask_validos.sum()
    eliminados = total_inicial - validados
    df.loc[~mask_validos, columna] = None

    # Reporte
    print("\n" + "="*60)
    print(f"{'REPORTE DE CALIDAD: CORREOS ELECTRÓNICOS':^60}")
    print("="*60)
    print(f"Registros iniciales:                 {total_inicial}")
    print(f"Correos válidos:                     {validados}")
    print(f"Correos eliminados (falsos/malos):   {eliminados}")
    print(f"Efectividad:                         {(validados/total_inicial)*100:.2f}%" if total_inicial > 0 else "N/A")
    print("-" * 60)
    
    return df

# Ejecución
df_consolidado = normalizar_y_validar_correos(df_consolidado)

## Nombre municipio

In [ ]:
# 1. Crear una "Lookup Table" (Tabla de búsqueda)
# Esto indexa los nombres de municipios por su código, facilitando el acceso instantáneo
mapeo_nombres = df_codigo_dane.set_index('CODIGO')['Nombre Municipio']

# 2. Aplicar el mapeo (Operación puramente vectorizada)
# Pandas busca cada valor de 'MNC_ID' en el índice de 'mapeo_nombres'
df_consolidado['Nombre_Municipio'] = df_consolidado['MNC_ID'].map(mapeo_nombres)

# --- ANÁLISIS DE RESULTADOS ---

# 3. Contar registros con y sin éxito
resumen_nulos = df_consolidado['Nombre_Municipio'].isna().value_counts()
sin_nombre = resumen_nulos.get(True, 0)
con_nombre = resumen_nulos.get(False, 0)

# 4. Frecuencia y porcentaje (Vectorizado)
conteo = df_consolidado['Nombre_Municipio'].value_counts(dropna=False)
porcentaje = df_consolidado['Nombre_Municipio'].value_counts(dropna=False, normalize=True) * 100

# Mostrar resultados
print(f"Cruce completado.\nExitosos: {con_nombre}\nFallidos: {sin_nombre}")
print("\nDistribución por Municipio:")
print(pd.concat([conteo, porcentaje], axis=1, keys=['Cantidad', 'Porcentaje (%)']))

del df_codigo_dane  # Liberar memoria del DataFrame de códigos DANE

## Asunto y correo copia capresocaeps@capresoca-casanare.gov.co 

In [ ]:
# 1. Preparar la fecha y nombres de meses en español
meses_es = {
    1: 'ENERO', 2: 'FEBRERO', 3: 'MARZO', 4: 'ABRIL', 
    5: 'MAYO', 6: 'JUNIO', 7: 'JULIO', 8: 'AGOSTO', 
    9: 'SEPTIEMBRE', 10: 'OCTUBRE', 11: 'NOVIEMBRE', 12: 'DICIEMBRE'
}

# Convertimos a datetime (ajusta el formato si es necesario, e.g., 'dayfirst=True')
fecha_dt = pd.to_datetime(df_consolidado['FECHA_PROCESO'], dayfirst=True)
mes_texto = fecha_dt.dt.month.map(meses_es)
anio_texto = fecha_dt.dt.year.astype(str)

# 2. Definir la condición de correo válido
# Filtra: No nulo, no vacío, no es "0", no es un espacio en blanco
mask_correo_valido = (
    df_consolidado['correo_electronico'].notna() & 
    (df_consolidado['correo_electronico'].astype(str).str.strip() != "") & 
    (df_consolidado['correo_electronico'].astype(str).str.strip() != "0")
)

# 3. Inicializar las columnas nuevas como vacías
df_consolidado['CC-copia'] = ""
df_consolidado['Asunto'] = ""

# 4. Asignar "CC-copia" a los que cumplen la máscara
df_consolidado.loc[mask_correo_valido, 'CC-copia'] = "capresocaeps@capresoca-casanare.gov.co"

# 5. Construir el "Asunto" dinámico solo para los registros válidos
# Definimos el tipo de trámite según el ESTADO_TRASLADO
tipo_tramite = df_consolidado['ESTADO_TRASLADO'].replace({
    'NEGADO': 'NEGADOS',
    'APROBADO': 'APROBADOS'
})

# Armamos el string usando concatenación vectorizada (muy rápida)
asunto_dinamico = (
    "NOTIFICACIÓN DE TRASLADOS " + tipo_tramite + 
    " A CAPRESOCA EPS EN EL MES DE " + mes_texto + 
    " " + anio_texto + " " + 
    df_consolidado['MNC_ID'].astype(str) + "_" + df_consolidado['Nombre_Municipio'].astype(str)
)

# 6. Asignar el asunto generado solo a las filas que cumplen la condición de correo
df_consolidado.loc[mask_correo_valido, 'Asunto'] = asunto_dinamico.loc[mask_correo_valido]

## Correos EPS origen "Negado"

In [ ]:
import pandas as pd
import numpy as np

def cruzar_correos_eps_negados(df_consolidado, df_eps):
    """
    Cruza los correos de las EPS hacia una NUEVA columna en el consolidado.
    Aplica solo a traslados NEGADOS donde el usuario ya tiene un correo válido.
    """
    print("Iniciando el cruce seguro de correos para traslados NEGADOS...")

    # ==========================================
    # PASO 1: PREPARAR EL DATAFRAME DE EPS (df_eps)
    # ==========================================
    df_eps.columns = df_eps.columns.str.strip()
    
    # Separamos múltiples códigos y "explotamos" en filas individuales
    df_eps['COD ENTIDAD'] = df_eps['COD ENTIDAD'].astype(str).str.split(r'[,;|-]+')
    df_eps_explotado = df_eps.explode('COD ENTIDAD')
    df_eps_explotado['COD ENTIDAD'] = df_eps_explotado['COD ENTIDAD'].str.strip().str.upper()

    # Agrupamos por código para unir correos únicos con ';'
    df_agrupado = df_eps_explotado.groupby('COD ENTIDAD').agg({
        'CORREO': lambda x: '; '.join([correo.strip() for correo in x.dropna().unique() if str(correo).strip() != '']),
        'ENTIDAD': 'first'
    }).reset_index()

    diccionario_correos = dict(zip(df_agrupado['COD ENTIDAD'], df_agrupado['CORREO']))
    diccionario_entidades = dict(zip(df_agrupado['COD ENTIDAD'], df_agrupado['ENTIDAD']))

    # ==========================================
    # PASO 2: DOBLE FILTRO Y CRUCE SEGURO
    # ==========================================
    df_consolidado['ENT_ID_ORIGEN'] = df_consolidado['ENT_ID_ORIGEN'].astype(str).str.strip().str.upper()
    df_consolidado['ESTADO_TRASLADO'] = df_consolidado['ESTADO_TRASLADO'].astype(str).str.strip().str.upper()

    # Condición A: Que el estado sea NEGADO
    condicion_negado = df_consolidado['ESTADO_TRASLADO'] == 'NEGADO'

    # Condición B: Que el correo del usuario sea válido (no sea 0, vacío, nulo, etc.)
    valores_invalidos = ['0', '', ' ', 'NAN', 'NULL', 'NONE']
    temp_correos_usuario = df_consolidado['correo_electronico'].astype(str).str.strip().str.upper()
    
    # Es válido si NO está en la lista de inválidos Y no es un nulo real (NaN)
    condicion_correo_valido = ~temp_correos_usuario.isin(valores_invalidos) & df_consolidado['correo_electronico'].notna()

    # Unimos ambas condiciones (el símbolo & significa "Y")
    mask_final = condicion_negado & condicion_correo_valido

    # ==========================================
    # PASO 3: ASIGNACIÓN A NUEVAS COLUMNAS
    # ==========================================
    # Inicializamos las nuevas columnas vacías (NaN) para toda la base
    df_consolidado['correo_eps'] = np.nan
    df_consolidado['nombre_eps'] = np.nan

    # Solo en las filas que cumplen la mask_final, mapeamos los valores
    df_consolidado.loc[mask_final, 'correo_eps'] = df_consolidado.loc[mask_final, 'ENT_ID_ORIGEN'].map(diccionario_correos)
    df_consolidado.loc[mask_final, 'nombre_eps'] = df_consolidado.loc[mask_final, 'ENT_ID_ORIGEN'].map(diccionario_entidades)

    print(f"Cruce completado. Se buscaron correos de EPS para {mask_final.sum()} registros que cumplían ambas condiciones.")
    
    return df_consolidado

# Ejecución
df_consolidado = cruzar_correos_eps_negados(df_consolidado, df_eps)

del df_eps  # Liberar memoria del DataFrame de EPS después del cruce

## Columna 'AIE_120' y organización decendente 

#### Acciones Realizadas
* **Creación de Columna de Control:** Se integró la columna `AIE_120` inicializada como cadena vacía (`""`). Esta columna está destinada a recibir los números de radicado de **Ventanilla Única**, funcionando como llave de trazabilidad externa.
* **Estrategia de Segmentación por Asunto:** Se aplicó un ordenamiento descendente sobre la columna `Asunto`. Dado que en Python las cadenas de texto tienen mayor peso en ordenamiento que una cadena vacía (`"NOTIFICACIÓN..." > ""`), esto desplaza automáticamente a los usuarios sin correo electrónico (que no tienen asunto generado) al final del DataFrame.

#### Justificación de Decisiones
* **Uso de `sort_values` con `ascending=False`:** Es el método más eficiente de Pandas (vectorizado en C) para mover registros "incompletos" al final sin necesidad de crear columnas auxiliares de "Peso" o "Ranking", ahorrando ciclos de CPU y memoria.
* **Manejo de Nulos:** Se incluyó `na_position='last'` como medida de seguridad redundante para asegurar que, si existiese algún valor `NaN` accidental en la columna Asunto, este no interrumpa el flujo de trabajo de notificación masiva.

#### Variables Gestionadas
* **Persisten:** `df_consolidado` (con la nueva estructura y orden).
* **Eliminadas/Liberadas:** La máscara temporal `mask_vacio` (liberada por el scope de la función) y referencias internas de ordenamiento mediante `gc.collect()`.

#### Advertencias
* **Volumen de Datos:** El ordenamiento es una operación $O(n \log n)$. Para el volumen actual de traslados (cientos o pocos miles), la ejecución es instantánea. Si el volumen superara los 500,000 registros, se recomendaría categorizar la columna 'Asunto' antes de ordenar.

In [ ]:
def procesar_organizacion_notificaciones(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crea la columna de radicado y organiza el DataFrame priorizando registros 
    con Asunto definido para la gestión de notificaciones.
    """
    # 1. Creación de columna para radicado externo (Ventanilla Única)
    # Se inicializa como Object (string) para soportar alfanuméricos futuros
    df['AIE_120'] = ""

    # 2. Lógica de Ordenamiento por Prioridad de Notificación
    # El objetivo es que los registros con 'Asunto' (listos para enviar) queden arriba.
    # Los vacíos o NaN deben quedar al final.
    
    # Creamos una serie temporal booleana: True (1) si está vacío, False (0) si tiene contenido
    # Al ordenar de forma ascendente, los 0 (con asunto) irán primero.
    mask_vacio = df['Asunto'].fillna("").str.strip() == ""
    
    # Ordenamiento multinivel:
    # Primero: Por presencia de Asunto (Los que tienen contenido primero)
    # Segundo: Por Estado (Aprobados vs Negados si se desea orden interno)
    # Tercero: Por Municipio (Para facilitar la revisión visual)
    df.sort_values(
        by=['Asunto', 'ESTADO_TRASLADO'],
        ascending=[False, True], # False en Asunto pone strings sobre vacíos ""
        inplace=True,
        na_position='last'
    )

    return df

# Ejecución del proceso
df_consolidado = procesar_organizacion_notificaciones(df_consolidado)

# Limpieza de memoria: No hay DFs temporales pesados, 
# pero forzamos recolección de basura de objetos de ordenamiento internos.
gc.collect()

## Unificar Correos

#### Explicación Técnica
Este bloque consolida las direcciones de correo del afiliado (`correo_electronico`) y de las entidades de salud (`correo_eps`) en un solo campo denominado **'Correos envio'**. El objetivo es generar una "cola de destinatarios" en una única celda por fila, permitiendo que las herramientas de automatización de oficina (Word/Outlook) y el personal de Ventanilla Única procesen los envíos y radicados de forma masiva sin necesidad de copiar datos de múltiples columnas.

#### Decisiones de Optimización y Calidad
* **Vectorización de Strings:** En lugar de usar un bucle `for` o `.apply(axis=1)`, se utilizó la suma directa de Series de Pandas. Esta es la operación más rápida en Python para manipulación de texto en DataFrames, reduciendo el tiempo de ejecución a milisegundos incluso con miles de registros.
* **Tratamiento de Nulos:** Se implementó `.fillna('')` antes de la concatenación. Sin esto, la presencia de un `NaN` en cualquiera de las columnas resultaría en un valor nulo para toda la celda unificada, causando pérdida de datos crítica.
* **Limpieza Estructural:** Se aplicó un `strip("; ")` para garantizar que si un registro solo posee correo de usuario o solo de EPS, la celda no comience ni termine con un separador innecesario que podría invalidar el envío en servidores SMTP.

#### Variables Gestionadas
* **Persisten:** `df_consolidado` (actualizado con la nueva columna de envío). Las columnas originales se mantienen intactas según el requerimiento de trazabilidad.
* **Eliminadas:** `col_usuario` y `col_eps` (series temporales liberadas automáticamente al terminar la función). Se llamó a `gc.collect()` para asegurar la liberación de memoria inmediata.

#### Advertencias
* **Separadores en Word:** Se utilizó `; ` como estándar. Si la configuración regional de su sistema de correspondencia requiere específicamente solo coma (`,`), se debe ajustar el carácter en la función de limpieza final.

In [ ]:
def unificar_correos_correspondencia(df: pd.DataFrame) -> pd.DataFrame:
    """
    Unifica las columnas de correo del usuario y correo de las EPS en una 
    sola columna 'Correos envio', optimizada para combinación de 
    correspondencia en Word y gestión de Ventanilla Única.
    """
    
    # 1. Preparación de datos (Vectorizado)
    # Convertimos a string y manejamos nulos para evitar concatenar "nan"
    # El strip() asegura que espacios en blanco no generen separadores huérfanos
    col_usuario = df['correo_electronico'].fillna('').astype(str).str.strip()
    col_eps = df['correo_eps'].fillna('').astype(str).str.strip()

    # 2. Unificación con separador ';'
    # Sumamos las series. Si ambas tienen datos, quedarán como: "correo@user.com; correo@eps.com"
    df['Correos envio'] = col_usuario + "; " + col_eps

    # 3. Limpieza de calidad de la cadena final (Regex y Strip)
    # - strip("; "): Elimina el punto y coma si solo una de las dos columnas tenía datos
    # - replace: Corrige casos donde pudieran quedar dos puntos y coma seguidos
    df['Correos envio'] = (
        df['Correos envio']
        .str.strip("; ")
        .str.replace(r";\s*;", "; ", regex=True)
    )

    # 4. Validación de Integridad (Auditoría rápida)
    conteo_vacios = (df['Correos envio'] == "").sum()
    print(f"--- Auditoría de Unificación ---")
    print(f"Registros con destinatarios: {len(df) - conteo_vacios}")
    print(f"Registros sin ningún correo: {conteo_vacios}")
    print(f"--------------------------------")

    return df

# Ejecución del bloque lógico
df_consolidado = unificar_correos_correspondencia(df_consolidado)

# Optimización de RAM: Eliminamos referencias locales y forzamos recolección
gc.collect()

In [ ]:
df_consolidado.columns

# Guardar resultados

#### Acciones Realizadas
* **Integración de Columna de Captura:** Se añadió la columna `AIE-120` como el último campo en la hoja **'VENTANILLA UNICA'**. Esta columna se entrega vacía pero estructurada para que el área administrativa asigne los radicados externos directamente.
* **Filtro de Gestión Operativa:** Se filtraron los registros para que Ventanilla Única solo reciba usuarios que posean al menos un correo electrónico válido en la columna `Correos envio`. Esto evita la generación de radicados para notificaciones que no se pueden enviar.
* **Enriquecimiento Visual (UX):** Mediante el motor `xlsxwriter`, se aplicó un formato de celda diferenciado (color amarillo suave) a la columna de radicados para que el usuario final identifique visualmente dónde debe ingresar la información.

#### Justificación de Decisiones
* **Selección Dinámica de Columnas:** Se implementó una validación de listas (`cols_existentes`) para asegurar que el script no falle si por algún motivo una columna técnica falta en el DataFrame principal, garantizando la continuidad del proceso.
* **Uso de Copias Locales:** El uso de `.copy()` en `df_vu` previene cualquier fragmentación de memoria y asegura que los formatos aplicados en el Excel no tengan efectos secundarios sobre el DataFrame `df_consolidado` que permanece en la RAM del notebook.

#### Variables Gestionadas
* **Persisten:** `df_consolidado` (Maestro de datos).
* **Eliminadas/Liberadas:** `df_aprobados`, `df_negados` y `df_vu` son eliminados al finalizar la exportación. Se ejecuta `gc.collect()` para liberar los objetos de estilo y estructuras de datos temporales del motor de Excel.

#### Advertencias
* **Integridad del Radicado:** Es vital instruir a Ventanilla Única para que no altere el orden de las filas en el archivo entregado, de modo que el cruce de retorno (de Excel al Notebook) mediante el número de identificación sea 100% preciso.

In [ ]:
def exportar_resultados_finales(
    df: pd.DataFrame, 
    periodo: str, 
    ruta_base: str
) -> None:
    """
    Exporta el consolidado a Excel con tres hojas optimizadas.
    Incluye la columna 'AIE-120' al final de la hoja de Ventanilla Única
    para la captura de radicados externos.
    """
    
    # --- 1. Configuración de Nombre y Ruta ---
    try:
        fecha_dt = pd.to_datetime(periodo, dayfirst=True)
        meses_es = {
            1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 
            5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto", 
            9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
        }
        nombre_archivo = f"Traslados Entrada {meses_es[fecha_dt.month]} {fecha_dt.year}.xlsx"
        
        if not os.path.exists(ruta_base):
            os.makedirs(ruta_base)
            
        ruta_final = os.path.join(ruta_base, nombre_archivo)
    except Exception as e:
        print(f"❌ Error en rutas: {e}")
        return

    # --- 2. Preparación de Segmentos (Hojas 1 y 2) ---
    # Bases completas para uso interno de la EPS
    df_aprobados = df[df['ESTADO_TRASLADO'].str.upper() == 'APROBADO'].copy()
    df_negados = df[df['ESTADO_TRASLADO'].str.upper() == 'NEGADO'].copy()

    # --- 3. Preparación Hoja 'VENTANILLA UNICA' (Hoja 3) ---
    # Filtro: Solo registros con destinatario de correo para evitar radicados inútiles
    mask_correo = (df['Correos envio'].notna()) & (df['Correos envio'].astype(str).str.strip() != "")
    
    # Definición del orden de columnas solicitado: Identificación -> Comunicación -> Radicado
    cols_vu = [
        'TPS_IDN_ID', 
        'HST_IDN_NUMERO_IDENTIFICACION', 
        'Asunto', 
        'Correos envio',
        'AIE-120' # Columna de captura para Ventanilla Única
    ]
    
    # Selección segura: verificamos existencia de columnas para evitar errores de ejecución
    cols_existentes = [c for c in cols_vu if c in df.columns]
    df_vu = df.loc[mask_correo, cols_existentes].copy()

    # --- 4. Escritura en Excel con XlsxWriter ---
    try:
        with pd.ExcelWriter(ruta_final, engine='xlsxwriter') as writer:
            # Exportación de datos
            df_aprobados.to_excel(writer, sheet_name='APROBADOS', index=False)
            df_negados.to_excel(writer, sheet_name='NEGADOS', index=False)
            df_vu.to_excel(writer, sheet_name='VENTANILLA UNICA', index=False)
            
            # Formateo visual para Ventanilla Única
            workbook  = writer.book
            worksheet = writer.sheets['VENTANILLA UNICA']
            
            # Formato de cabecera resaltada
            header_fmt = workbook.add_format({'bold': True, 'bg_color': '#C6E0B4', 'border': 1})
            # Formato específico para la columna de captura (amarillo claro para indicar "llenar aquí")
            input_fmt = workbook.add_format({'bg_color': '#FFF2CC', 'border': 1, 'italic': True})
            
            # Aplicar formatos a las cabeceras
            for col_num, value in enumerate(df_vu.columns.values):
                worksheet.write(0, col_num, value, header_fmt)
            
            # Resaltar visualmente la columna AIE-120 (última columna)
            idx_aie = len(df_vu.columns) - 1
            worksheet.set_column(idx_aie, idx_aie, 20, input_fmt)

        print("\n" + "="*60)
        print(f"✅ EXPORTACIÓN EXITOSA CON CAPTURA DE RADICADOS")
        print(f"Archivo: {nombre_archivo}")
        print(f"Registros VU para radicar: {len(df_vu)}")
        print("="*60)

    except Exception as e:
        print(f"❌ Error al guardar Excel: {e}")
    finally:
        # --- 5. Limpieza de Memoria ---
        del df_aprobados, df_negados, df_vu
        gc.collect()

# Ejecución
exportar_resultados_finales(df_consolidado, Periodo_notificación, R_salida_base)